## Load the dataset

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# Download the dataset
!wget https://www.webpages.uidaho.edu/vakanski/Codes_Data/spam_messages.csv

--2025-03-28 19:50:21--  https://www.webpages.uidaho.edu/vakanski/Codes_Data/spam_messages.csv
Resolving www.webpages.uidaho.edu (www.webpages.uidaho.edu)... 129.101.105.230
Connecting to www.webpages.uidaho.edu (www.webpages.uidaho.edu)|129.101.105.230|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 45108305 (43M) [application/octet-stream]
Saving to: ‘spam_messages.csv.1’

spam_messages.csv.1 100%[===================>]  43.02M  29.3MB/s    in 1.5s    

2025-03-28 19:50:23 (29.3 MB/s) - ‘spam_messages.csv.1’ saved [45108305/45108305]



In [ ]:
# Load the CSV file
dataset = pd.read_csv('spam_messages.csv', index_col=False, encoding = 'latin-1')

# Display the shape of the dataset
print('Data shape', dataset.shape)

# Display the first 10 messages in the dataset
dataset.head(10)

Data shape (47392, 2)


,label,text
0,ham,"Funny fact Nobody teaches volcanoes 2 erupt, t..."
1,ham,I sent my scores to sophas and i had to do sec...
2,spam,We know someone who you know that fancies you....
3,ham,Only if you promise your getting out as SOON a...
4,spam,Congratulations ur awarded either ï¿½500 of CD...
5,ham,"I'll text carlos and let you know, hang on"
6,ham,K.i did't see you.:)k:)where are you now?
7,ham,No message..no responce..what happend?
8,ham,Get down in gandhipuram and walk to cross cut ...
9,ham,You flippin your shit yet?


In [ ]:
# Replace 0 with 'ham' and 1 with 'spam' in the labels
dataset['label'] = dataset['label'].replace({'ham':0, 'spam':1})

dataset

<ipython-input-19-b094fd578c8d>:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset['label'] = dataset['label'].replace({'ham':0, 'spam':1})


,label,text
0,0,"Funny fact Nobody teaches volcanoes 2 erupt, t..."
1,0,I sent my scores to sophas and i had to do sec...
2,1,We know someone who you know that fancies you....
3,0,Only if you promise your getting out as SOON a...
4,1,Congratulations ur awarded either ï¿½500 of CD...
...,...,...
47387,0,please change the sitara tickets dating back t...
47388,1,from mr . silas sankoh ecobank benin rue du go...
47389,0,just to get things stirred up : the estate wil...
47390,0,sorry for the oversight . bill - - - - - origi...


In [ ]:
# Define a list with class labels
class_names = ['ham', 'spam']

In [ ]:
# Display fully the first few messages (can you guess which one is spam without looking at the label?)
print(dataset.iloc[0,1])
print(dataset.iloc[1,1])
print(dataset.iloc[2,1])
print(dataset.iloc[3,1])
print(dataset.iloc[5,1])

Funny fact Nobody teaches volcanoes 2 erupt, tsunamis 2 arise, hurricanes 2 sway aroundn no 1 teaches hw 2 choose a wife Natural disasters just happens
I sent my scores to sophas and i had to do secondary application for a few schools. I think if you are thinking of applying, do a research on cost also. Contact joke ogunrinde, her school is one me the less expensive ones
We know someone who you know that fancies you. Call 09058097218 to find out who. POBox 6, LS15HB 150p
Only if you promise your getting out as SOON as you can. And you'll text me in the morning to let me know you made it in ok.
I'll text carlos and let you know, hang on


In [ ]:
# Install the datasets library from Hugging Face
!pip install -q datasets fsspec

In [ ]:
from datasets import Dataset

# Split the dataset into train, test, and validation sets
train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)
train_data, val_data = train_test_split(dataset, test_size=0.1, random_state=42)

# create the datasets from pandas dataframes
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)
test_dataset = Dataset.from_pandas(test_data)

# remove the index column to retain only label and text columns
train_dataset = train_dataset.remove_columns(["__index_level_0__"])
val_dataset  = val_dataset .remove_columns(["__index_level_0__"])
test_dataset = test_dataset.remove_columns(["__index_level_0__"])

In [ ]:
# Check the train dataset
train_dataset

Dataset({
    features: ['label', 'text'],
    num_rows: 42652
})

## Prepare the data for model fitting

In [ ]:
from transformers import AutoTokenizer

# Tokenizers in Hugging Face are functions that convert input text into numerical format

# Load the tokenizer for the model Distilbert
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# Create a function to use the loaded tokenizer and convert the text into numercial format
def tokenize(rows):
  return tokenizer(rows['text'], padding="max_length",
truncation=True, max_length=128)

In [ ]:
# Tokenize the training, validation, and test datasets
tokenized_train_dataset = train_dataset.map(tokenize, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize, batched=True)
tokenized_test_dataset =  test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/42652 [00:00<?, ? examples/s]

Map:   0%|          | 0/4740 [00:00<?, ? examples/s]

Map:   0%|          | 0/9479 [00:00<?, ? examples/s]

## Import a pretrained model

In [ ]:
from transformers import AutoModelForSequenceClassification

# Import the Distilbert model and specify the number of classes
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import pipeline

fill_mask = pipeline(
    "fill-mask",
    model="google/mobilebert-uncased",
    tokenizer="google/mobilebert-uncased"
)

print(
    fill_mask(f"HuggingFace is creating a {fill_mask.tokenizer.mask_token} that the community uses to solve NLP tasks.")
)


config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/147M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/147M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cuda:0


[{'score': 0.1204613447189331, 'token': 6994, 'token_str': 'tool', 'sequence': 'huggingface is creating a tool that the community uses to solve nlp tasks.'}, {'score': 0.09160231798887253, 'token': 4132, 'token_str': 'platform', 'sequence': 'huggingface is creating a platform that the community uses to solve nlp tasks.'}, {'score': 0.0906333401799202, 'token': 2291, 'token_str': 'system', 'sequence': 'huggingface is creating a system that the community uses to solve nlp tasks.'}, {'score': 0.05025921389460564, 'token': 2944, 'token_str': 'model', 'sequence': 'huggingface is creating a model that the community uses to solve nlp tasks.'}, {'score': 0.0442996583878994, 'token': 7705, 'token_str': 'framework', 'sequence': 'huggingface is creating a framework that the community uses to solve nlp tasks.'}]


## Train the model

In [ ]:
from transformers import TrainingArguments

# Define training arguments for the model
training_args = TrainingArguments(
    # Directory to save model checkpoints and logs
    output_dir="distilbert-emotion",
    # Evaluation strategy - evaluate the model at the end of each epoch
    eval_strategy="epoch",
    # Learning rate for the optimizer
    learning_rate=2e-5,
    # Batch size for training
    per_device_train_batch_size=64,
    # Batch size for evaluation
    per_device_eval_batch_size=64,
    # Number of training epochs
    num_train_epochs=2,
    # Weight decay to apply for regularization
    weight_decay=0.01,
    # Disable reporting to external tools (e.g., WandB, TensorBoard)
    report_to="none"
)

In [ ]:
from sklearn.metrics import accuracy_score

# Function to compute metrics during training
def compute_metrics(eval_pred):
    outputs, labels = eval_pred
    predictions = outputs.argmax(axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

In [ ]:
from transformers import Trainer

# Initialize the Trainer with the required parameters
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
# Fit the model (about 4 minutes)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.132500,0.052931,0.981435
2,0.047800,0.046429,0.985021


TrainOutput(global_step=1334, training_loss=0.07576646404466529, metrics={'train_runtime': 944.872, 'train_samples_per_second': 90.281, 'train_steps_per_second': 1.412, 'total_flos': 2824999743737856.0, 'train_loss': 0.07576646404466529, 'epoch': 2.0})

## Evaluate the Model

# Report the classification accuracy for the test dataset. For full marks, it is expected to report test accuracy above 97%

In [ ]:
# Evaluate the model on the test dataset
test_metrics = trainer.evaluate(eval_dataset=tokenized_test_dataset)

# Print the accuracy
print(test_metrics)

{'eval_loss': 0.03240102529525757, 'eval_accuracy': 0.9902943348454478, 'eval_runtime': 36.2967, 'eval_samples_per_second': 261.153, 'eval_steps_per_second': 4.105, 'epoch': 2.0}


## Predict the class of several test sentences

In [ ]:
# List of sentences to evaluate
test_texts = ["You won an award! It is a free cruise trip! Call us immediately to claim it.", "Free apple phones on us for our customers, call immediately!.", "I read a great book yesterday.", "March Madness has been really fun to watch this year."]

# Tokenize the sentences, pad them to the same length for batch processing
inputs = tokenizer(test_texts, return_tensors="pt", padding=True, truncation=True).to(device=0)
inputs

{'input_ids': tensor([[  101,  2017,  2180,  2019,  2400,   999,  2009,  2003,  1037,  2489,
          8592,  4440,   999,  2655,  2149,  3202,  2000,  4366,  2009,  1012,
           102],
        [  101,  2489,  6207, 11640,  2006,  2149,  2005,  2256,  6304,  1010,
          2655,  3202,   999,  1012,   102,     0,     0,     0,     0,     0,
             0],
        [  101,  1045,  3191,  1037,  2307,  2338,  7483,  1012,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0],
        [  101,  2233, 12013,  2038,  2042,  2428,  4569,  2000,  3422,  2023,
          2095,  1012,   102,     0,     0,     0,     0,     0,     0,     0,
             0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 

In [ ]:
# Forward pass through the model
outputs = model(**inputs)

# Get predicted labels for each sentence
predicted_labels = outputs.logits.argmax(axis=-1).cpu().tolist()
print("Predicted Labels:", predicted_labels)

Predicted Labels: [1, 1, 0, 0]


# Report the predicted class (spam or ham) by the model for the four messages in Step 2.

In [ ]:
# Print results
for text, label in zip(test_texts, predicted_labels):
    print(f"Text: '{text}'; Predicted Label: {class_names[label]}")

Text: 'You won an award! It is a free cruise trip! Call us immediately to claim it.'; Predicted Label: spam
Text: 'Free apple phones on us for our customers, call immediately!.'; Predicted Label: spam
Text: 'I read a great book yesterday.'; Predicted Label: ham
Text: 'March Madness has been really fun to watch this year.'; Predicted Label: ham


# Generate adversarial examples

In [ ]:
# Install the TextAttack library
!pip install -qq textattack

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 40.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 38.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.7/445.7 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Download the required resource for the NLTK (Natural Language Toolkit) library
import nltk
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
# Wrap the trained model and tokenizer for Distilbert into a textattack model
from textattack.models.wrappers import HuggingFaceModelWrapper

wrapped_model = HuggingFaceModelWrapper(model, tokenizer)

textattack: Updating TextAttack package dependencies.
textattack: Downloading NLTK required packages.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw to /root/nltk_data...
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


## Apply TextFooler attack

In [ ]:
# TextFooler attack: replaces words with synonyms to fool the model
from textattack.attack_recipes import TextFoolerJin2019

attack_1 = TextFoolerJin2019.build(wrapped_model)

textattack: Downloading https://textattack.s3.amazonaws.com/word_embeddings/paragramcf.
100%|██████████| 481M/481M [00:14<00:00, 32.9MB/s]
textattack: Unzipping file /root/.cache/textattack/tmp6d0lw28_.zip to /root/.cache/textattack/word_embeddings/paragramcf.
textattack: Successfully saved word_embeddings/paragramcf to cache.
textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


# Step 1: Apply the TextFooler attack on the following message: Free apple phones on us for our customers, call us immediately!

In [ ]:
from textattack.datasets import Dataset
from textattack import Attacker

# Example text sentence
example_text = "Free apple phones on us for our customers, call us immediately!"
# Correct label for the text sentence: class label 1 is 'joy'
example_label = 1

# Create a new dataset with the example text sentence and label
# The dataset is simply a list of text and label pairs
example_dataset = Dataset([[example_text, example_label]])

In [ ]:
# Apply the attack
attacker = Attacker(attack_1, example_dataset)
attacker.attack_dataset()

textattack: Attempting to attack 10 samples when only 1 are available.


Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.5
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (1): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
    (2): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.840845057
        (window_size):  15
        (skip_text_shorter_than_window):  True
        (compare_against_original):  False
      )
    (3): RepeatModification
    (4): StopwordModification
    (5): InputColumnModification(
        (matching_column_labels):  ['premise', 'hypothesis']
       

 10%|█         | 1/10 [00:25<03:49, 25.51s/it]

--------------------------------------------- Result 1 ---------------------------------------------


[Succeeded / Failed / Skipped / Total] 1 / 0 / 0 / 1:  10%|█         | 1/10 [00:26<03:56, 26.31s/it]

[[1 (100%)]] --> [[0 (61%)]]

Free [[apple]] phones on us for our [[customers]], call [[us]] [[immediately]]!

Free [[mitt]] phones on us for our [[diners]], call [[we]] [[soon]]!



+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 1      |
| Number of failed attacks:     | 0      |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 0.0%   |
| Attack success rate:          | 100.0% |
| Average perturbed word %:     | 36.36% |
| Average num. words per input: | 11.0   |
| Avg num queries:              | 80.0   |
+-------------------------------+--------+


# Step 2: Select the first 10 messages from the test dataset, and apply the TextFooler attack to generate adversarial examples.

In [ ]:
# Select the first 10 samples from the test dataset to create adverasarial exampples
text_samples = test_dataset["text"][:10]
label_samples = test_dataset["label"][:10]

In [ ]:
# check the data
text_samples

['im working technical support voice process',
 'dirty pictureblyk on aircel thanks you for being a valued member heres an exclusive animated trailer for the dirty picture the insider scoop saysvidya balans sexy avatar itsy bitsy clothes cleavageshowing blouses and raunchy dance numbers in the dirty picture are receiving jawdropping reactions fans and trade pundits say the masala movie is expected to beat the musical treat rockstar the funfilled desi boyz and the action thriller don 2 dont miss it!',
 'has your mortgage search got you down are you frustrated and confused with all the different terms and quotes don t know who is telling you the truth we can solve all your problems visit our site today and in two minutes you can have us searching thousands of programs and lenders for you get the truth get the facts get your options all in one shot it s absolutely free and you can be done in only two minutes so hyperlink click right now and put your worries behind you ryte 1635465 po16354

In [ ]:
# check the true labels
label_samples

[0, 1, 1, 1, 1, 0, 0, 1, 1, 0]

In [ ]:
# Recall the class names
class_names

['ham', 'spam']

In [ ]:
# Convert the data to the required format for TextAttack
testdataset10samples_dataset = Dataset(list(zip(text_samples, label_samples)))

# Run the attack
attacker = Attacker(attack_1, testdataset10samples_dataset)
attacker.attack_dataset()

Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.5
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (1): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
    (2): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.840845057
        (window_size):  15
        (skip_text_shorter_than_window):  True
        (compare_against_original):  False
      )
    (3): RepeatModification
    (4): StopwordModification
    (5): InputColumnModification(
        (matching_column_labels):  ['premise', 'hypothesis']
       

[Succeeded / Failed / Skipped / Total] 1 / 0 / 0 / 1:  10%|█         | 1/10 [00:01<00:16,  1.78s/it]

--------------------------------------------- Result 1 ---------------------------------------------
[[0 (100%)]] --> [[1 (82%)]]

[[im]] [[working]] [[technical]] support voice [[process]]

[[lim]] [[artworks]] [[artistic]] support voice [[cure]]




[Succeeded / Failed / Skipped / Total] 2 / 0 / 0 / 2:  20%|██        | 2/10 [00:03<00:14,  1.81s/it]

--------------------------------------------- Result 2 ---------------------------------------------
[[1 (96%)]] --> [[0 (52%)]]

dirty pictureblyk on aircel thanks you for being a valued member [[heres]] an [[exclusive]] animated trailer for the dirty picture the insider scoop saysvidya balans sexy avatar itsy bitsy clothes cleavageshowing blouses and raunchy dance numbers in the dirty picture are receiving jawdropping reactions fans and trade pundits say the masala movie is expected to beat the musical treat rockstar the funfilled desi boyz and the action thriller don 2 dont miss it!

dirty pictureblyk on aircel thanks you for being a valued member [[thats]] an [[mere]] animated trailer for the dirty picture the insider scoop saysvidya balans sexy avatar itsy bitsy clothes cleavageshowing blouses and raunchy dance numbers in the dirty picture are receiving jawdropping reactions fans and trade pundits say the masala movie is expected to beat the musical treat rockstar the funfilled de

...

--------------------------------------------- Result 8 ---------------------------------------------
[[1 (100%)]] --> [[[FAILED]]]

n 45days reduce fat8 inchusesauna slim belt650 vibration belt999 ncr50rs extrafree yoko call now:9268250573/931130487




[Succeeded / Failed / Skipped / Total] 8 / 1 / 0 / 9:  90%|█████████ | 9/10 [01:17<00:08,  8.60s/it]

--------------------------------------------- Result 9 ---------------------------------------------
[[1 (100%)]] --> [[0 (50%)]]

googlecash gives you all the tools you need to turn the search engine google . com into an autopilot [[cash]] generating machine ! what ' s your dream [[lifestyle]] ? [[challenge]] schnabel behind jolla cozy buttrick isabel [[solder]] excerpt breakfast tame firewall hatchet [[ooze]] boggle [[decompression]] london recriminate coat inbreed churchyard dar yoke snappy extrema grind cookie longitudinal [[suntan]] buttress [[herpes]] quadrillion improvisate idiot antarctica maybe [[embarrass]] bermuda [[lawman]] octillion [[musket]] betoken [[alacrity]] [[request]] [[seizure]] fateful [[mile]] conjunct courier administrable there stepmother boycott importune chronicle cavern catch aeronautic [[prestigious]] thermofax interruption irredeemable hesitater [[pursue]] abbas aruba berra dean lockwood [[jimenez]] sioux [[ac]] collocation [[dempsey]] bowstring maintenan

[Succeeded / Failed / Skipped / Total] 9 / 1 / 0 / 10: 100%|██████████| 10/10 [01:18<00:00,  7.89s/it]

--------------------------------------------- Result 10 ---------------------------------------------
[[0 (99%)]] --> [[1 (64%)]]

hi [[does]] anyone have a collection of real world spam that i could [[use]] to test my setup this url email is sponsored by osdn tired of that same old cell phone get a new here for free url spamassassin talk mailing list spamassassin talk url url

hi [[want]] anyone have a collection of real world spam that i could [[resorted]] to test my setup this url email is sponsored by osdn tired of that same old cell phone get a new here for free url spamassassin talk mailing list spamassassin talk url url



+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 9      |
| Number of failed attacks:     | 1      |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 10.0%  |
| Attack success r

***TEXTFOOLER Attack Success Rate: 90%***

# Step 3: Apply the WordBug attack on the first 10 messages from the test dataset to generate adversarial examples

In [ ]:
# WordBug attack introduces character-level perturbations

from textattack.attack_recipes import DeepWordBugGao2018
attack_2 = DeepWordBugGao2018.build(wrapped_model)

textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


In [ ]:
# Run the attack to the 10 sentences from the test dataset
attacker = Attacker(attack_2, testdataset10samples_dataset)
attacker.attack_dataset()

Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  CompositeTransformation(
    (0): WordSwapNeighboringCharacterSwap(
        (random_one):  True
      )
    (1): WordSwapRandomCharacterSubstitution(
        (random_one):  True
      )
    (2): WordSwapRandomCharacterDeletion(
        (random_one):  True
      )
    (3): WordSwapRandomCharacterInsertion(
        (random_one):  True
      )
    )
  (constraints): 
    (0): LevenshteinEditDistance(
        (max_edit_distance):  30
        (compare_against_original):  True
      )
    (1): RepeatModification
    (2): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 1 / 0 / 0 / 1:  10%|█         | 1/10 [00:00<00:05,  1.77it/s]

--------------------------------------------- Result 1 ---------------------------------------------
[[0 (100%)]] --> [[1 (78%)]]

[[im]] [[working]] [[technical]] [[support]] [[voice]] [[process]]

[[iEm]] [[worknig]] [[techical]] [[spport]] [[voie]] [[proess]]




[Succeeded / Failed / Skipped / Total] 2 / 0 / 0 / 2:  20%|██        | 2/10 [00:01<00:07,  1.09it/s]

--------------------------------------------- Result 2 ---------------------------------------------
[[1 (96%)]] --> [[0 (60%)]]

dirty pictureblyk on aircel thanks you for being a [[valued]] member heres an [[exclusive]] animated trailer for the dirty picture the insider scoop saysvidya balans [[sexy]] avatar itsy bitsy clothes cleavageshowing blouses and raunchy dance numbers in the [[dirty]] picture are [[receiving]] jawdropping reactions fans and trade pundits say the masala movie is expected to [[beat]] the musical treat rockstar the funfilled desi [[boyz]] and the action thriller don 2 dont miss it!

dirty pictureblyk on aircel thanks you for being a [[vadlued]] member heres an [[xeclusive]] animated trailer for the dirty picture the insider scoop saysvidya balans [[sey]] avatar itsy bitsy clothes cleavageshowing blouses and raunchy dance numbers in the [[irty]] picture are [[receving]] jawdropping reactions fans and trade pundits say the masala movie is expected to [[bat]] the m

...

--------------------------------------------- Result 8 ---------------------------------------------
[[1 (100%)]] --> [[[FAILED]]]

n 45days reduce fat8 inchusesauna slim belt650 vibration belt999 ncr50rs extrafree yoko call now:9268250573/931130487




[Succeeded / Failed / Skipped / Total] 6 / 3 / 0 / 9:  90%|█████████ | 9/10 [00:26<00:02,  2.95s/it]

--------------------------------------------- Result 9 ---------------------------------------------
[[1 (100%)]] --> [[[FAILED]]]

googlecash gives you all the tools you need to turn the search engine google . com into an autopilot cash generating machine ! what ' s your dream lifestyle ? challenge schnabel behind jolla cozy buttrick isabel solder excerpt breakfast tame firewall hatchet ooze boggle decompression london recriminate coat inbreed churchyard dar yoke snappy extrema grind cookie longitudinal suntan buttress herpes quadrillion improvisate idiot antarctica maybe embarrass bermuda lawman octillion musket betoken alacrity request seizure fateful mile conjunct courier administrable there stepmother boycott importune chronicle cavern catch aeronautic prestigious thermofax interruption irredeemable hesitater pursue abbas aruba berra dean lockwood jimenez sioux ac collocation dempsey bowstring maintenance orono extant vast debunk galvanometer punster swatch cryptology contain conv

[Succeeded / Failed / Skipped / Total] 7 / 3 / 0 / 10: 100%|██████████| 10/10 [00:27<00:00,  2.72s/it]

--------------------------------------------- Result 10 ---------------------------------------------
[[0 (99%)]] --> [[1 (73%)]]

hi does anyone have a collection of real world spam that i could use to test my setup this [[url]] email is sponsored by [[osdn]] tired of that same old cell phone get a new here for free url spamassassin talk mailing list spamassassin talk url url

hi does anyone have a collection of real world spam that i could use to test my setup this [[uDrl]] email is sponsored by [[sdn]] tired of that same old cell phone get a new here for free url spamassassin talk mailing list spamassassin talk url url



+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 7      |
| Number of failed attacks:     | 3      |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 30.0%  |
| Attack success rate: 

***WORDBUG ATTACK Success Rate: 70%***

# Step 4: Apply the PWWS (Probability Weighted Word Saliency) attack on the first 10 messages from the test dataset to generate adversarial examples.

In [ ]:
# PWWS attack uses word importance scores to modify text (i.e., modify the most important words in a sentence)
from textattack.attack_recipes import PWWSRen2019
attack_3 = PWWSRen2019.build(wrapped_model)

[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


In [ ]:
# Run the attack to the 10 sentences from the test dataset
attacker = Attacker(attack_3, testdataset10samples_dataset)
attacker.attack_dataset()

Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  weighted-saliency
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapWordNet
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 1 / 0 / 0 / 1:  10%|█         | 1/10 [00:02<00:25,  2.89s/it]

--------------------------------------------- Result 1 ---------------------------------------------
[[0 (100%)]] --> [[1 (60%)]]

im [[working]] technical support voice [[process]]

im [[cultivate]] technical support voice [[serve]]




[Succeeded / Failed / Skipped / Total] 2 / 0 / 0 / 2:  20%|██        | 2/10 [00:10<00:43,  5.49s/it]

--------------------------------------------- Result 2 ---------------------------------------------
[[1 (96%)]] --> [[0 (64%)]]

[[dirty]] pictureblyk on aircel thanks you for being a valued member [[heres]] an [[exclusive]] animated trailer for the dirty picture the insider scoop saysvidya balans sexy avatar itsy bitsy clothes cleavageshowing blouses and raunchy dance numbers in the dirty picture are receiving jawdropping reactions fans and trade pundits say the masala movie is expected to beat the musical treat rockstar the funfilled desi boyz and the action thriller don 2 dont miss it!

[[muddy]] pictureblyk on aircel thanks you for being a valued member [[Hera]] an [[scoop]] animated trailer for the dirty picture the insider scoop saysvidya balans sexy avatar itsy bitsy clothes cleavageshowing blouses and raunchy dance numbers in the dirty picture are receiving jawdropping reactions fans and trade pundits say the masala movie is expected to beat the musical treat rockstar the funf

...

--------------------------------------------- Result 8 ---------------------------------------------
[[1 (100%)]] --> [[[FAILED]]]

n 45days reduce fat8 inchusesauna slim belt650 vibration belt999 ncr50rs extrafree yoko call now:9268250573/931130487




[Succeeded / Failed / Skipped / Total] 6 / 3 / 0 / 9:  90%|█████████ | 9/10 [02:01<00:13, 13.46s/it]

--------------------------------------------- Result 9 ---------------------------------------------
[[1 (100%)]] --> [[[FAILED]]]

googlecash gives you all the tools you need to turn the search engine google . com into an autopilot cash generating machine ! what ' s your dream lifestyle ? challenge schnabel behind jolla cozy buttrick isabel solder excerpt breakfast tame firewall hatchet ooze boggle decompression london recriminate coat inbreed churchyard dar yoke snappy extrema grind cookie longitudinal suntan buttress herpes quadrillion improvisate idiot antarctica maybe embarrass bermuda lawman octillion musket betoken alacrity request seizure fateful mile conjunct courier administrable there stepmother boycott importune chronicle cavern catch aeronautic prestigious thermofax interruption irredeemable hesitater pursue abbas aruba berra dean lockwood jimenez sioux ac collocation dempsey bowstring maintenance orono extant vast debunk galvanometer punster swatch cryptology contain conv

[Succeeded / Failed / Skipped / Total] 7 / 3 / 0 / 10: 100%|██████████| 10/10 [02:06<00:00, 12.67s/it]

--------------------------------------------- Result 10 ---------------------------------------------
[[0 (99%)]] --> [[1 (94%)]]

[[hi]] does anyone have a collection of real world spam that i could [[use]] to test my setup this url email is sponsored by osdn tired of that same old cell phone get a new here for free url spamassassin talk mailing list spamassassin talk url url

[[how-do-you-do]] does anyone have a collection of real world spam that i could [[usance]] to test my setup this url email is sponsored by osdn tired of that same old cell phone get a new here for free url spamassassin talk mailing list spamassassin talk url url



+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 7      |
| Number of failed attacks:     | 3      |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 30.0%  |
| Attack 

***Success Rate: 70%***

# Step 05: Describe the above three adversarial attacks. Write at least three sentences for each of the attacks to explain the methods they use for generating adversarial examples.

1. **TextFooler Attack**  

   The TextFooler attack substitutes synonyms in a text to create adversarial examples with the restriction that the resulting sentence is still grammatically and semantically similar to the original. A model makes words towards the predicted classification output and looks for the most misleading synonym to make the substitution. The model classifies the text based on the changes that have been made to certain keywords, which leads to the classification being changed without the text losing its quality.

2. **WordBug Attack**  

   In the WordBug attack, the model manipulates a specific character from within a word by adding, deleting, or swapping characters to create small shabby words, which operate on the lower levels of the text. These perturbs are intended to create humanly readable but obfuscating to a text classification model. Whereas TextFooler’s synonym approach assumes the model relies on an exact character sequence, WordBug works on the opposite assumption, which makes it useful against models with poor spelling correction.

3. **PWWS Attack**

  The PWWS attack is an advance word-level adversarial attack that first calculates the contribution of every single word in the sentence to the model’s classification and subsequently seeks to modify the most salient one. Words are evaluated according to their contribution to the prediction and replaced with synonyms that are likely to enable the model to make the wrong classification. Unlike TextFooler, which strives to maintain semantic similarity, PWWS focuses on high-impact word manipulation while causing changes to the sentence structure.



##Step 06: Elaborate on the attacks on the MobileBERT model and the achieved success rate in the previous steps. Write your opinion on which of these attacks created adversarial messages that are the least perceptible by human users. Write between 5 to 10 sentences.

With the **Text Fooler** attack, the success rate reached a whopping 90 dismissing the models accuracy from 100 to 10 percent with only 20.5 percent of words being perturbed on average. It likely generated the least detectable adversarial messages due to its modification being less noticeable. Conversely, the **Wordbug** and **PWWS** attacks had a lower success rate of 70 and and involved changing a higher percentage of words (24.88 and 16.94 percent, respectively). These more drastic changes may turn out the alter the adversarial messages to such a level which makes them visible to human readers. Regardless, all three attacks were effective at fooling the MobileBERT model. Text Fooler most likely created the easiest to read and most sophisticatedly written messages.

